In [9]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
import pandas as pd
import protfasta
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob, os
from tqdm import tqdm
import json

# Plan: Remove overlap wtih synthetics
WE will evaluate model on synthetics. so removing from the training data

In [11]:
# The values were slightly updated when evaluating regression
# Version of the data had changes
# Seqs should be the same between the two files
synthetics_seqs = pd.read_csv("data/synthetics_DF.csv")
synthetics_seqs

,Unnamed: 0,AAseq,Activity,FACS_Activity,ActivityStdDev,StdActivityStdDev,NumReps,Charge,HydrophobicityKD,HydrophobicityWW,...,ADPred Score,State Size,# Models Predicted,# Models Predicted (ADPredNoKotha),# Models Predicted (all 5),TADA_Standard,PADDLE_Standard,# Models Predicted (Standard),# Models Predicted (Standard ADPredNoKotha),# Models Predicted (Standard all 5)
0,0,AAAAAIACELRTDNYFEYPDLSILDALDNDSDETMSLIDE,0.079859,25075.335066,NaN,NaN,1,-10.0,0.484167,-0.33600,...,0.592839,3.0,2.0,1.0,2.0,1,1,3.0,2.0,3.0
1,1,AAAAASDHSDVNDIIELGGLSNRLYDFYSDLEDIENDIAV,0.150008,42429.450978,NaN,NaN,1,-9.0,0.487222,-0.32275,...,0.684129,3.0,3.0,3.0,3.0,1,1,3.0,3.0,3.0
2,2,AAADGASFSDDTDEDLYKYDPWDLPGLSVIKTDDPEWFFF,0.655776,167550.966119,10429.712428,0.036511,4,-9.0,0.440000,-0.19475,...,0.705396,5.0,4.0,3.0,4.0,1,1,4.0,3.0,4.0
3,3,AAADGASFSDDTDEDLYKYDPWELPSKASFGEQEWYFFSP,0.420822,109426.029848,12411.816570,0.043450,4,-8.0,0.404167,-0.22175,...,0.568411,5.0,3.0,2.0,3.0,1,0,3.0,2.0,3.0
4,4,AAADGPDCDTREFCELRDSLLWNESCFLLDDDKRFDVSEN,0.205442,56143.316707,NaN,NaN,1,-8.0,0.422500,-0.37475,...,0.642051,4.0,3.0,2.0,3.0,1,1,4.0,3.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16765,17272,YYSDDIMLDLGVDLSNFQFDFRYFGLDSPLSPPWNEDDDS,1.008116,254716.006477,6276.949290,0.020717,3,-9.0,0.434444,-0.12200,...,0.796153,3.0,4.0,3.0,4.0,1,1,4.0,3.0,4.0
16766,17273,YYSEQSCYLDLDLDLDITFLDFSQFLEQLLQSKNDPLVSE,0.939006,237618.847801,32880.537293,0.108521,3,-8.0,0.474167,-0.16700,...,0.766623,4.0,4.0,3.0,4.0,1,1,4.0,3.0,4.0
16767,17274,YYSEQSCYLNFPPLDCVDLVATLSEKFKSFSQLLLDFPIS,0.164664,46055.201492,NaN,NaN,1,-3.0,0.520833,-0.04550,...,0.655498,5.0,1.0,1.0,1.0,1,0,2.0,2.0,2.0
16768,17275,YYSEQSCYLNFPPLDMFPALPSDLSGWVESMLSELNNPAS,0.551329,141711.824042,NaN,NaN,2,-5.0,0.478333,-0.07375,...,0.704097,5.0,2.0,2.0,2.0,1,1,3.0,3.0,3.0


In [5]:
protfasta.write_fasta(dict(zip(synthetics_seqs["Unnamed: 0"], synthetics_seqs["AAseq"])), "data/synthetics.fasta")

Terminal commands to do clustering: 

mmseqs createdb /global/scratch/projects/fc_mvslab/OpenProjects/Sanjana/consensus_AD_predictor/data/train_seqs_with_test_overlap_removed.fasta train_seqs_with_test_overlap_removedDB

mmseqs createdb /global/scratch/projects/fc_mvslab/OpenProjects/Sanjana/consensus_AD_predictor/data/synthetics.fasta syntheticsDB

mmseqs search syntheticsDB train_seqs_with_test_overlap_removedDB remove_synthetics tmp --cov-mode 2 -c 0.9 --min-seq-id 0.3 -s 7 --remove-tmp-files 1

mmseqs convertalis syntheticsDB train_seqs_with_test_overlap_removedDB remove_synthetics train_seqs_with_test_synthetics_overlap_removed.tsv --format-output "query,target,pident,qcov,tcov,evalue,bits"

In [6]:
mmseqs_search_result = pd.read_csv("data/mmseqs/train_seqs_with_test_synthetics_overlap_removed.tsv", sep = "\t", header = None)
mmseqs_search_result.columns = ["query","target","pident","qcov","tcov","evalue","bits"]
mmseqs_search_result

,query,target,pident,qcov,tcov,evalue,bits
0,1061,AT5G61420.1,47.1,0.900,0.136,8.029000e-04,37
1,11827,Lj2g3v1985040.1,69.6,0.900,0.106,1.588000e-09,53
2,11827,XP_010940544.1,53.4,0.950,0.055,1.702000e-05,42
3,16110,XP_010542182.1,84.6,0.975,0.058,9.359000e-16,71
4,16110,Lsa003913,82.3,1.000,0.081,1.287000e-15,70
...,...,...,...,...,...,...,...
23963,5958,Achn289291,64.1,0.975,0.118,3.008000e-09,52
23964,5958,Neem_5630_f_1,64.1,0.975,0.116,3.008000e-09,52
23965,5958,Ahy022629,63.7,0.975,0.112,4.141000e-09,52
23966,5958,Thhalv10021174m,66.1,0.925,0.118,5.699000e-09,51


In [7]:
train_TFs_to_remove = mmseqs_search_result["target"].value_counts().index
train_TFs_to_remove

Index(['KFK37763.1', 'XP_010525037.1', 'PDK_30s1076031g001', 'Spipo17G0034500',
       'AA40G00382', 'XP_013626928.1', 'WALNUT_00021799-RA', 'XP_010549797.1',
       'Cla009127', 'Gorai.004G268400.1',
       ...
       'XP_010929381.1', 'WALNUT_00004727-RA', 'MA_109677g0010',
       'Niben101Scf04745g02009.1', 'Jcr4S14571.10', 'Peaxi162Scf00634g00032.1',
       'XP_010098434.1', 'GSVIVT01014287001', '27950.m000105',
       'mrna00971.1-v1.0-hybrid'],
      dtype='object', name='target', length=2495)

In [12]:
train_seqs = protfasta.read_fasta("data/train_seqs_with_test_overlap_removed.fasta")
train_seqs = pd.DataFrame({'id' : train_seqs.keys(), 'seq' : train_seqs.values()})
train_seqs

,id,seq
0,Q8GGH0,MESFLLSKVSFVIKKIRLEKGMTQEDLAYKSNLDRTYISGIERNSR...
1,F1R2J1,MHEETSNSTLQGKTSFSIADILDPAKFNGTRETREISNNRESPKTT...
2,O75364,MEFGLLSEAEARSPALSLSDAGTPHPQLPEHGCKGQEHSDSEKASA...
3,P09087,MQQHHLQQQQQQQQQQEQQHLQEQQQHLQQLHHHAHHHLPQPLHTT...
4,P43609,MSDTEKDKDVPMVDSHEATEEPPTTSTNTPSFPHLAQEQAKEESAT...
...,...,...
234779,g004716.m1_yHMPu5000040912_saturnispora_halmia...,MQQKRAFPSNEGPNGDVNENQHGETIIANQETDRRGGMERDSEEAL...
234780,g002397.m1_yHMPu5000041691_saccharomycopsis_cr...,MSKRPGEETEEDTIKKLKIESQQPKDDNNVIDSELLTHQTNYQDDG...
234781,g002544.m1_yHDO593_barnettozyma_sp_180604.final,MDNISTPNQMEEQYTYLEEPAIIIDQNGGESQGLLTEQQQQQQQQL...
234782,g003106.m1_yHMPu5000034601_starmera_dryadoides...,QQQQQQQQQQQQQQQQQQPPPPPPPQQQVSESPNDPQPVQEVAVVE...


In [13]:
train_subset_test_synthetics_overlap_removed = train_seqs[~train_seqs["id"].isin(list(train_TFs_to_remove))]
train_subset_test_synthetics_overlap_removed

,id,seq
0,Q8GGH0,MESFLLSKVSFVIKKIRLEKGMTQEDLAYKSNLDRTYISGIERNSR...
1,F1R2J1,MHEETSNSTLQGKTSFSIADILDPAKFNGTRETREISNNRESPKTT...
2,O75364,MEFGLLSEAEARSPALSLSDAGTPHPQLPEHGCKGQEHSDSEKASA...
3,P09087,MQQHHLQQQQQQQQQQEQQHLQEQQQHLQQLHHHAHHHLPQPLHTT...
4,P43609,MSDTEKDKDVPMVDSHEATEEPPTTSTNTPSFPHLAQEQAKEESAT...
...,...,...
234779,g004716.m1_yHMPu5000040912_saturnispora_halmia...,MQQKRAFPSNEGPNGDVNENQHGETIIANQETDRRGGMERDSEEAL...
234780,g002397.m1_yHMPu5000041691_saccharomycopsis_cr...,MSKRPGEETEEDTIKKLKIESQQPKDDNNVIDSELLTHQTNYQDDG...
234781,g002544.m1_yHDO593_barnettozyma_sp_180604.final,MDNISTPNQMEEQYTYLEEPAIIIDQNGGESQGLLTEQQQQQQQQL...
234782,g003106.m1_yHMPu5000034601_starmera_dryadoides...,QQQQQQQQQQQQQQQQQQPPPPPPPQQQVSESPNDPQPVQEVAVVE...


In [25]:
protfasta.write_fasta(dict(zip(train_subset_test_synthetics_overlap_removed["id"], train_subset_test_synthetics_overlap_removed["seq"])), "../data/train_seqs_with_test_synthetics_overlap_removed.fasta")

In [15]:
# What are the train seqs which cluster with synthetics closely?
train_seqs[train_seqs["id"].isin(list(train_TFs_to_remove))]

,id,seq
4278,Achn235071,MDVYGASVTECFRVDDLLDFSNDEIFSGTATADSDLHRLPPSGNTY...
4284,Achn202561,MLRLGTKERVRGGSVEGCGTRVHLIRGERERKVRERKMMGSDFMDE...
4287,Achn266441,MERKLGVLFRVFSLFLPRTNNCLGSMTVPEDASCFGDGGCWPPGFR...
4310,Aco011895.1,MAPVSLPPGFRFHPTDEELIIYYLKRKINGRKIELEIIPEVDLYKC...
4311,Aco001109.1,YIGYLEESLGAMEEKSDHVDKSDEVMLPGFRFHPTDEELVGFYLKR...
...,...,...
231785,Cotton_A_25450_BGI-A2_v1.0,MKGELELPPGFRFHPTDDELVNHYLCRKCASQPISVPIIAEIDLYK...
231789,Cotton_A_12652_BGI-A2_v1.0,MEDTIFSQGTMLGGLPDSAMDFDYMDELLLDGCWLETTQGSDFLNP...
231794,Cotton_A_01569_BGI-A2_v1.0,MESCVPPGFRFHPTDEELVGYYLRKKIASKKIDLDVIRDIDLYRIE...
231819,Gh_A02G0977,MSEDMNLSINGQSQVPPGFRFHPTEEELLHYYLRKKVAYEKIDLDV...
